In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names, supported FBref leagues, and season configuration
# are loaded from 00_project_config.
#
# Reusable reading, cleaning, validation, deduplication, and writing helpers
# are loaded from 01_spark_utils.

SILVER_TABLE = f"{SILVER_SCHEMA}.team_goalkeeping_stats"

FBREF_DATASET = "goalkeeping_stats"

# One Silver row represents:
#
#   team + season + league
#
# for the FBref team goalkeeping dataset.
BUSINESS_KEY = [
    "team_name",
    "season_code",
    "league_code",
]

REQUIRED_SILVER_COLUMNS = [
    "team_name",
    "team_key",
    "season_code",
    "season_start_year",
    "league_code",
]


# -----------------------------------------------------------------------------
# Build expected Bronze table list
# -----------------------------------------------------------------------------
# FBref Bronze maintains one table per:
#
#   dataset + league
#
# Each Bronze table contains all supported seasons for that league.

BRONZE_TABLES = [
    f"{BRONZE_SCHEMA}.fbref_{FBREF_DATASET}_{league_code}"
    for league_code in FBREF_LEAGUES
]


# -----------------------------------------------------------------------------
# Read and combine Bronze tables
# -----------------------------------------------------------------------------
# read_and_union_tables() is strict by default:
#
#   - every expected league table must exist
#   - schemas must be compatible
#
# A missing league should fail the pipeline rather than silently create an
# incomplete Silver dataset.

bronze_df = read_and_union_tables(
    spark,
    BRONZE_TABLES,
)


# -----------------------------------------------------------------------------
# Validate expected FBref source columns
# -----------------------------------------------------------------------------
# FBref is an external HTML source and its table structure can change.
# Validate the columns we depend on before performing transformations so schema
# drift fails with a clear error message.

REQUIRED_BRONZE_COLUMNS = [
    "Squad",
    "#_Pl",
    "Playing_Time_MP",
    "Playing_Time_Starts",
    "Playing_Time_Min",
    "Playing_Time_90s",
    "Performance_GA",
    "Performance_GA90",
    "Performance_SoTA",
    "Performance_Saves",
    "Performance_Save%",
    "Performance_W",
    "Performance_D",
    "Performance_L",
    "Performance_CS",
    "Performance_CS%",
    "Penalty_Kicks_PKatt",
    "Penalty_Kicks_PKA",
    "Penalty_Kicks_PKsv",
    "Penalty_Kicks_PKm",
    "Penalty_Kicks_Save%",
    "_season",
    "_league",
    "_stat",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------
# FBref Bronze is already flat, so the shared top-level string cleaner is
# appropriate here.
#
# It trims values and converts empty strings to NULL while preserving the
# original Bronze column names.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate dataset identity
# -----------------------------------------------------------------------------
# All rows loaded through this notebook should originate from the FBref
# goalkeeping dataset.
#
# This check protects against accidental table/path configuration mistakes.

unexpected_stat_rows = (
    bronze_df
    .filter(
        F.col("_stat").isNull()
        | (F.col("_stat") != F.lit(FBREF_DATASET))
    )
    .limit(1)
    .count()
)

if unexpected_stat_rows > 0:
    raise ValueError(
        f"Unexpected _stat value found while processing {FBREF_DATASET}."
    )

# -----------------------------------------------------------------------------
# Validate FBref league identifiers
# -----------------------------------------------------------------------------
# Bronze retains source-specific league codes (pl, ll, bl, sa, lo).
# Before converting them into canonical platform identifiers, verify
# that every source code is supported by the project configuration.
#
# Fail explicitly instead of allowing unknown codes to become NULL
# during the mapping transformation.

invalid_league_rows = (
    bronze_df
    .filter(
        F.col("_league").isNull()
        | ~F.col("_league").isin(*FBREF_LEAGUES.keys())
    )
    .limit(1)
    .count()
)

if invalid_league_rows > 0:
    raise ValueError(
        f"Unsupported FBref league identifier found in {FBREF_DATASET}. "
        f"Expected one of: {sorted(FBREF_LEAGUES.keys())}."
    )

print("FBref source league identifiers validated.")

# -----------------------------------------------------------------------------
# Select, rename, and cast Silver columns
# -----------------------------------------------------------------------------
# Source-oriented FBref names are converted into clear analytical names.
#
# Important semantic corrections:
#
#   Performance_SoTA
#       -> shots_on_target_against
#
#   Penalty_Kicks_PKatt
#       -> penalty_kicks_faced
#
#   Penalty_Kicks_PKA
#       -> penalty_kicks_allowed
#
# These names more accurately reflect the underlying FBref metrics.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        clean_string("Squad")
        .alias("team_name"),

        F.col("`#_Pl`")
        .cast("int")
        .alias("goalkeepers_used_count"),


        # ---------------------------------------------------------------------
        # Playing time
        # ---------------------------------------------------------------------

        F.col("Playing_Time_MP")
        .cast("int")
        .alias("matches_played"),

        F.col("Playing_Time_Starts")
        .cast("int")
        .alias("starts"),

        F.col("Playing_Time_Min")
        .cast("int")
        .alias("minutes_played"),

        F.col("Playing_Time_90s")
        .cast("double")
        .alias("nineties_played"),


        # ---------------------------------------------------------------------
        # Goalkeeping performance
        # ---------------------------------------------------------------------

        F.col("Performance_GA")
        .cast("int")
        .alias("goals_against"),

        F.col("Performance_GA90")
        .cast("double")
        .alias("goals_against_per_90"),

        F.col("Performance_SoTA")
        .cast("int")
        .alias("shots_on_target_against"),

        F.col("Performance_Saves")
        .cast("int")
        .alias("saves"),

        F.col("`Performance_Save%`")
        .cast("double")
        .alias("save_percentage"),

        F.col("Performance_W")
        .cast("int")
        .alias("wins"),

        F.col("Performance_D")
        .cast("int")
        .alias("draws"),

        F.col("Performance_L")
        .cast("int")
        .alias("losses"),

        F.col("Performance_CS")
        .cast("int")
        .alias("clean_sheets"),

        F.col("`Performance_CS%`")
        .cast("double")
        .alias("clean_sheet_percentage"),


        # ---------------------------------------------------------------------
        # Penalty-kick performance
        # ---------------------------------------------------------------------

        F.col("Penalty_Kicks_PKatt")
        .cast("int")
        .alias("penalty_kicks_faced"),

        F.col("Penalty_Kicks_PKA")
        .cast("int")
        .alias("penalty_kicks_allowed"),

        F.col("Penalty_Kicks_PKsv")
        .cast("int")
        .alias("penalty_kicks_saved"),

        F.col("Penalty_Kicks_PKm")
        .cast("int")
        .alias("penalty_kicks_missed"),

        F.col("`Penalty_Kicks_Save%`")
        .cast("double")
        .alias("penalty_kick_save_percentage"),


        # ---------------------------------------------------------------------
        # Season / league context
        # ---------------------------------------------------------------------
        # Keep the original FBref source identifiers for lineage and matching.
        #
        # season_start_year provides a normalized project-wide season value:
        #
        #   2122 -> 2021
        #   2425 -> 2024

        F.col("_season")
        .cast("string")
        .alias("season_code"),

        season_code_to_start_year("_season")
        .alias("season_start_year"),

        # Normalize FBref source league identifiers to canonical values.
        # The mapping is defined centrally in 00_project_config.
        map_values(
            "_league",
            FBREF_LEAGUES,
        ).alias("league_code"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)



# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Business-key validation happens before deduplication so NULL team/season/
# league values cannot be grouped together and accidentally reduced to a
# single arbitrary record.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so rerunning the same FBref Bronze load
# can create multiple versions of the same team/season/league record.
#
# Keep the record from the latest Bronze ingestion.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Resolve canonical team key
# -----------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_name(
    spark,
    SILVER_SCHEMA,
    "fbref",
)

silver_df = resolve_team_keys_by_name(
    silver_df,
    team_mappings_df,
    source_name_column="team_name",
    output_key_column="team_key",
    fail_on_unmapped=True,
)

print("FBref canonical team resolution completed.")

# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# The helper validates the required analytical identifiers and also provides
# the final row count without requiring a separate count() action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team goalkeeping records into {SILVER_TABLE}."
)